# Set ROIs

This notebook allows you to make ROIs for a video without going through the entire pipeline. The example frame used as the image to set the ROIs is fetched using OpenCV's video reader.\
The functionality of this notebook is technically subsumed by the basic pipeline notebook.

**If your data is local**: Just specify the path to the video you want to draw ROIs on.

**If your data is on a server**: OpenCV's video reader allows for you to read just a single frame from a video file without transferring the entire thing IF the server directory is mounted. So we recommend mounting the server directory if possible to avoid transferring the entire file. This can be done with:
- Windows: https://support.microsoft.com/en-us/windows/map-a-network-drive-in-windows-29ce55d1-34e3-a7e2-4801-131475f9557d
- OSX: https://www.google.com/search?q=mount+network+drive+osx
- Linux: Use RClone CLI: `rclone mount remote:path/to/files /path/to/local/mount`
    - Example: `rclone mount transfer:/n/files/Neurobio/MICROSCOPE/ /mnt/MICROSCOPE/`
    

In [ ]:
# ALWAYS RUN THIS CELL
# widen jupyter notebook window
from IPython.display import display, HTML
display(HTML("<style>.container {width:95% !important; }</style>"))

%load_ext autoreload
%autoreload 2
import face_rhythm as fr
import face_rhythm.alignment

from pprint import pprint
from pathlib import Path
import copy
import getpass

import cv2

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import natsort

try:
    import bnpm
except:
    pass

fr.util.system_info(verbose=True);

## Set paths

In [ ]:
use_remote = False  ## set to True if running on remote machine. Will use SFTP to find files and load frames

In [ ]:
if use_remote:
    import bnpm
    import getpass
    username=input("Enter your username: ")
    password=getpass.getpass(prompt="Enter your password: ")

    use_localSshKey = False
    path_sshKey = '/home/rich/.ssh/id_rsa' if use_localSshKey else None
    
    ssh_t = bnpm.server.ssh_interface(
        nbytes_toReceive=20000,
        recv_timeout=1,
        verbose=True,
    )
    ssh_t.o2_connect(
        hostname='transfer.rc.hms.harvard.edu',
        username=username,
        password=password if (use_localSshKey==False) else None,
        key_filename=path_sshKey,
        look_for_keys=False,
        passcode_method=1,
        verbose=0,
        skip_passcode=False,    
    )
    # ssh_t.fasrc_connect(
    #     hostname='login.rc.fas.harvard.edu',
    #     username=input('Username: '),
    #     password=getpass.getpass(prompt='Password: ') if (use_localSshKey==False) else None,
    #     verbose=0,
    # )
    sftp_t = bnpm.server.sftp_interface(ssh_client=ssh_t.client)

In [ ]:
directory_videos  = '/n/netscratch/bsabatini_lab/Lab/rhakim/data/2pRAM/facerhythm_stroke_biomarker_exp/camera'
filename_strMatch = 'video4.*'  ## You can use regular expressions to search and match more complex strings
strMatch_in_path = 'PS50'

In [ ]:
directory_save = r'/n/netscratch/bsabatini_lab/Lab/rhakim/analysis/face_rhythm/ROI_alignment/cam4/PS50/'

In [ ]:
if use_remote:
    paths_videos = sftp_t.search_recursive(
        dir_outer=directory_videos, 
        reMatch=filename_strMatch, 
#         reMatch_in_path=strMatch_in_path,
        depth=5,
        find_files=True,
        find_folders=True,
        natsorted=True,
        verbose=True
    )
else:
    paths_videos = fr.helpers.find_paths(
        dir_outer=directory_videos,
        reMatch=filename_strMatch,  ## string to use to search for files in directory. Uses regular expressions!alg_ns=
        reMatch_in_path=strMatch_in_path,
        depth=5,  ## how many folders deep to search
        verbose=True,
    )

display(paths_videos)

In [ ]:
names_sessions = natsort.natsorted([Path(p).parts[-2].replace('-', '') for p in paths_videos])
# names_sessions = natsort.natsorted([bnpm.path_helpers.find_date_in_path(p).replace('-', '') for p in paths_videos])
names_sessions

In [ ]:
pipeline = fr.alignment.Image_preparation_pipeline(
    ds_factor=20,
    ptile_specVar_keep=10,
    ptile_intensity_keep=100,
    params_vqt={
        "Fs_sample": 250,
        "Q_lowF": 3.5,
        "Q_highF": 20,
        "F_min": 0.5,
        "F_max": 60,
        "n_freq_bins": 50,
        "window_type": 'hann',
        "downsample_factor": 10,
        "fft_conv": True,
        "plot_pref": False,
    },
    clip_limit=2.0,
    grid_size=60,
    verbose=True,
)

## Fetch image

In [ ]:
# idx_template = 5

In [ ]:
# path_vid_template = paths_videos[idx_template]
# path_vid_template

#### Using a remote server
- (if your videos are on the local computer, then skip past this step)
- Make sure ffmpeg is installed if you want to import first frames from a remote server. It is not necessary otherwise

In [ ]:
# if use_remote:
#     # Replace with your actual connection details and remote video path.
#     extractor = fr.alignment.SFTPVideoFrameExtractor(
#         host="transfer.rc.hms.harvard.edu",
#         username=username,
#         password=password,
#         port=22,  # or your specific port
#     )

In [ ]:
# time_start = 0  # Start time in seconds
# duration = 20  # Duration to capture in seconds

# if use_remote:
#     # Specify the path to the remote video file.
#     im_test = np.array(extractor.extract_frames(path_vid_template, time_start=time_start, duration=duration), dtype=np.float32).mean(-1)
# else:
#     im_test = fr.alignment.get_frames(path_vid_template, time_start, time_start + duration)

In [ ]:
# bnpm.misc.estimate_array_size(im_test)

In [ ]:
# im_aug = pipeline.apply_pipeline(torch.as_tensor(im_test, dtype=torch.float32).mean(-1).numpy())
# im_aug.shape

# Define ROIs

Either select new ROIs (`select_mode='gui'`), or import existing ROIs (`path_file=path_to_ROIs.h5_file`).\
Typically, you should make 1 or 2 ROIs. One for defining where the face points should be and one for cropping the frame.

In [ ]:
# im_aug = np.load(r'/Users/richardhakim/Documents/data/facerhythm_stroke_biomarker_exp/analysis/face_rhythm/aligned_dots/cam1/im_template.npy')

In [ ]:
# np.save(r'/Users/richardhakim/Documents/data/facerhythm_stroke_biomarker_exp/analysis/face_rhythm/aligned_dots/cam1/im_template.npy', im_aug)

In [ ]:
%matplotlib widget
rois = fr.rois.ROIs(
    # select_mode='gui',
    # exampleImage=im_aug,
    select_mode='file',
    # path_file=r'/n/netscratch/bsabatini_lab/Lab/rhakim/analysis/face_rhythm/ROI_alignment/cam3/ROIs.h5',
    path_file=r'/n/netscratch/bsabatini_lab/Lab/rhakim/analysis/face_rhythm/ROI_alignment/cam4/ROIs.h5',
    verbose=2
)

Save the `ROIs` object in the 'analysis_files' project folder

In [ ]:
# rois.fliplr()

In [ ]:
# %matplotlib inline
# rois.make_points(rois[0], point_spacing=14) if rois.point_positions is None else None
rois.plot_rois()

In [ ]:
path_save = str(Path(directory_save) / 'ROIs.h5')
rois.save_run_data(path_run_data=path_save, overwrite=True, verbose=1)

# Optional: Multisession alignment

The below code shows how to align the points from one 'template' session onto multiple other 'new' sessions.\
Steps:
1. Get example images from each session: `images`
2. Optionally adjust the local contrast of each example image to make: `images_toUse`
3. Instantiate the `ROI_Aligner` class and choose which OpenCV optical flow method to use: `aligner`
4. Perform non-rigid registration to warp the 'template' ROIs and points onto the example images from each session: `aligner.align_and_make_ROIs`
5. Retrieve the newly made `ROIs` class objects: `rois_objs_new = aligner.ROIs_objects_new`
6. Save the new `ROIs` class objects: `rois_objs_new[x].save_run_data()`
7. Visualize the results!

1. Get example images

In [ ]:
%matplotlib inline
if use_remote:
    images = {path: pipeline.apply_pipeline(extractor.extract_frames(path, time_start=0, duration=40)[:, :, :, 0]) for path in tqdm(paths_videos)}
else:
    images = {path: pipeline.apply_pipeline(fr.alignment.get_frames(path, time_start=0, time_end=40)[:, :, :, 0]) for path in tqdm(paths_videos[:])}

2. Optionally adjust local contrast

3. to 5. Register ROIs from template to new session images, then make new `ROIs` objects

In [ ]:
## Save images
(Path(directory_save) / 'images').mkdir(parents=True, exist_ok=True)
[np.save(str(Path(directory_save) / 'images' / f"{date}.npy"), im) for date, im in tqdm(zip(names_sessions, images.values()))]
;

In [ ]:
# images = {date: np.load(str(Path(directory_save) / 'images' / f"{date}.npy")) for date in names_sessions}

In [ ]:
ims_aug = [im.astype(np.float32) for im in images.values()]
ia_max = np.max(np.stack(ims_aug, axis=0))
ims_aug = [im / ia_max for im in ims_aug]

In [ ]:
im_template = rois.exampleImage

im_aug_template = pipeline.apply_clahe(((im_tmp:=(im_template**0.5)) / im_tmp.max()), clip_limit=2.0, grid_size=25,)

im_aug_template = (im_aug_template.astype(np.float32) / im_aug_template.astype(np.float32).max())

In [ ]:
plt.figure()
plt.imshow(im_aug_template, cmap='gray')

In [ ]:
# # im_aug_template = ims_aug[idx_template]
# im_aug_template = np.load("/n/netscratch/bsabatini_lab/Lab/rhakim/analysis/face_rhythm/ROI_alignment/cam1/PS50/im_template.npy")

In [ ]:
fr.visualization.display_toggle_image_stack([im_aug_template] + ims_aug)

In [ ]:
## If the roicat library is installed, import it
%load_ext autoreload
%autoreload 2
try:
    import roicat
    import roicat.tracking.alignment
except ImportError:
    print("roicat not installed. Using default alignment method.")
    roicat = None
    pass

In [ ]:
import skimage

In [ ]:
fr.visualization.display_toggle_image_stack([skimage.filters.gaussian(im, sigma=.0)[40:-320, 100:-100] for im in ([im_aug_template,] + ims_aug[:])])

In [ ]:
if roicat is not None:
    aligner = roicat.tracking.alignment.Aligner(
        use_match_search=True,
        all_to_all=True,
        radius_in=15,
        radius_out=75,
        order=5,
        z_threshold=50,
        device='cuda:0' if torch.cuda.is_available() else 'cpu',
        verbose=2,
    )

    aligner.fit_geometric(
        # template=0.5,  ## specifies which image to use as the template. Either array (image), integer (ims_moving index), or float (ims_moving fractional index)
        template=0,
        # ims_moving=[im_aug_template[:,:,0],] + ims_aug,
        ims_moving=[skimage.filters.gaussian(im, sigma=4.0) for im in ([im_aug_template[:,:,0],] + ims_aug[:])],
        template_method='image',  ## 'sequential': align images to neighboring images (good for drifting data). 'image': align to a single image
        # mask_borders=(40,320, 100,100),  ## number of pixels to mask off the edges (top, bottom, left, right)
        mask_borders=(0,50, 0,0),  ## number of pixels to mask off the edges (top, bottom, left, right)
        method='RoMa',  ## See below for options.
        kwargs_method = {
            'RoMa': {  ## Accuracy: Best, Speed: Very slow (can be fast with a GPU).
                'model_type': 'outdoor',
                'n_points': 10000,  ## Higher values mean more points are used for the registration. Useful for larger FOV_images. Larger means slower.
                'batch_size': 1000,
            },
            'DISK_LightGlue': {  ## Accuracy: Good, Speed: Fast.
                'num_features': 3000,  ## Number of features to extract and match. I've seen best results around 2048 despite higher values typically being better.
                'threshold_confidence': 0.0,  ## Higher values means fewer but better matches.
                'window_nms': 7,  ## Non-maximum suppression window size. Larger values mean fewer non-suppressed points.
            },
            'LoFTR': {  ## Accuracy: Okay. Speed: Medium.
                'model_type': 'indoor_new',
                'threshold_confidence': 0.2,  ## Higher values means fewer but better matches.
            },
            'ECC_cv2': {  ## Accuracy: Okay. Speed: Medium.
                'mode_transform': 'euclidean',  ## Must be one of {'translation', 'affine', 'euclidean', 'homography'}. See cv2 documentation on findTransformECC for more details.
                'n_iter': 200,
                'termination_eps': 1e-09,  ## Termination criteria for the registration algorithm. See documentation for more details.
                'gaussFiltSize': 1,  ## Size of the gaussian filter used to smooth the FOV_image before registration. Larger values mean more smoothing.
                'auto_fix_gaussFilt_step': 10,  ## If the registration fails, then the gaussian filter size is reduced by this amount and the registration is tried again.
            },
            'PhaseCorrelation': {  ## Accuracy: Poor. Speed: Very fast. Notes: Only applicable for translations, not rotations or scaling.
                'bandpass_freqs': [1, 30],
                'order': 5,
            },
        },
        constraint='euclidean',
        kwargs_RANSAC = {
            'inl_thresh': 3.0,  ## cv2.findHomography RANSAC inlier threshold. Larger values mean more lenient matching.
            'max_iter': 100,
            'confidence': 0.99,
        },
        verbose=True,  ## Set to 3 to view plots of the alignment process if available for the method.
    )

    ims_aligned = aligner.transform_images(
        ims_moving=[im_aug_template[:,:,0],] + ims_aug,
        remappingIdx=aligner.remappingIdx_geo,
    )[1:]
    
    remappingIdx_geo = copy.deepcopy(aligner.remappingIdx_geo[1:])
    
    aligner.plot_alignment_results_geometric()
    
else:
    aligner = fr.rois.Image_Aligner(verbose=True)

    aligner.fit_geometric(
        template=im_aug_template,  ## template image
        # template=ims_aug[10],  ## template image
        ims_moving=ims_aug,  ## images to align
        template_method='image',  ## 'sequential': align images to neighboring images (good for drifting data). 'image': align to a single image
        mode_transform='euclidean',  ## type of geometric transformation. See openCV's cv2.findTransformECC for details
        # mask_borders=(0,150,150,50),  ## number of pixels to mask off the edges (top, bottom, left, right)
        mask_borders=(20,300,100,100),  ## number of pixels to mask off the edges (top, bottom, left, right)
        n_iter=200,  ## number of iterations for optimization
        termination_eps=1e-09,  ## convergence tolerance
        gaussFiltSize=11,  ## size of gaussian blurring filter applied to all images
        auto_fix_gaussFilt_step=10,  ## increment in gaussFiltSize after a failed optimization
    )

    aligner.transform_images_geometric(ims_aug);

    ims_aligned = copy.deepcopy(aligner.ims_registered_geo)
    
    remappingIdx_geo = copy.deepcopy(aligner.remappingIdx_geo)

In [ ]:
fr.visualization.display_toggle_image_stack([im_aug_template,] + ims_aligned)

In [ ]:
# aligner.fit_nonrigid(
#     template=im_template_geo,  ## template image
# #     template=int(0),  ## specifies which image to use as the template. Either array (image), integer (ims_moving index), or float (ims_moving fractional index)
#     ims_moving=aligner.ims_registered_geo,  ## Input images. Typically the geometrically registered images
#     remappingIdx_init=aligner.remappingIdx_geo,  ## The remappingIdx between the original images (and ROIs) and ims_moving
#     template_method='image',  ## 'sequential': align images to neighboring images. 'image': align to a single image, good if using geometric registration first
#     mode_transform='createOptFlow_DeepFlow',  ## algorithm for non-rigid transformation. Either 'createOptFlow_DeepFlow' or 'calcOpticalFlowFarneback'. See openCV docs for each. 
#     kwargs_mode_transform=None,  ## kwargs for `mode_transform`
# )

# aligner.transform_images_nonrigid(list(images.values()));

In [ ]:
fr.visualization.display_toggle_image_stack([im_aug_template] + aligner.ims_registered_geo)

# fr.visualization.display_toggle_image_stack([im_aug_template] + aligner.ims_registered_nonrigid)

In [ ]:
def transform_points(points, remappingIdx):
    ## Transform points using the remapping index
    points_remap = fr.helpers.remap_points(
        points=points,
        remappingIdx=remappingIdx,
        interpolation='linear',
        fill_value=None,
    )

    ## Clip points to image size
    points_remap[:, 0] = np.clip(points_remap[:, 0], 0, remappingIdx.shape[1] - 1)
    points_remap[:, 1] = np.clip(points_remap[:, 1], 0, remappingIdx.shape[0] - 1)

    return points_remap


In [ ]:
remappingIdx = copy.deepcopy(remappingIdx_geo)

points_roiBorders_transformed = [{key: transform_points(
    points=points,
    remappingIdx=rmap_idx,
) for key, points in rois.roi_points.items()} for rmap_idx in remappingIdx]

points_forTracking_transformed = [transform_points(
    points=rois.point_positions,
    remappingIdx=rmap_idx,
) for rmap_idx in remappingIdx]

# exampleImages = list(images.values())
exampleImages = [np.tile((e * (0.15)).astype(np.uint8)[..., None], (1, 1, 3)) for e in images.values()]

In [ ]:
rois_objs_new = {name: fr.rois.ROIs(
    select_mode='custom', 
    coords_rois=borderPoints, 
    exampleImage=exampleImage, 
    point_positions=point_positions
) for name, borderPoints, exampleImage, point_positions in tqdm(zip(paths_videos, points_roiBorders_transformed, exampleImages, points_forTracking_transformed))}

In [ ]:
frame_visualizer = fr.visualization.FrameVisualizer(
    display=False,
    frame_height_width=rois_objs_new[list(rois_objs_new.keys())[0]].img_hw,
    point_sizes=3,
    points_colors=(255,0,0),
    alpha=0.5,
)

images_with_warped_points = [frame_visualizer.visualize_image_with_points(
    image=rois.exampleImage,
    points=rois.point_positions,
) for rois in rois_objs_new.values()]

In [ ]:
fr.visualization.display_toggle_image_stack(images_with_warped_points)

## 6. Save
Save the new `ROIs` objects. These can be used to initialize the `ROIs` objects in each face-rhythm run.

In [ ]:
# names_sessions = ['__'.join(Path(name).parts[-6:-4]) for name in rois_objs_new.keys()]

for ii, (name, rois_new) in enumerate(rois_objs_new.items()):
    path_save = str(Path(directory_save) / names_sessions[ii] / f'ROIs.h5')
#     path_save = str(Path(directory_save) / f'ROIs.h5')
    print(path_save)
    rois_new.save_run_data(
        path_run_data=path_save,
        overwrite=True,
        verbose=1,
    )

## 7. Visualize!

In [ ]:
# fr.visualization.display_toggle_image_stack(aligner.ims_registered_nonrigid)

In [ ]:
# ## Cast to uint8
# movie = [np.tile((im * 0.15).astype(np.uint8)[:,:,None], (1,1,3)) for im in aligner.ims_registered_nonrigid]
# ## Add text overlay
# movie = fr.helpers.add_text_to_images(movie, [[str(n)] for n in np.arange(len(movie))], position=(50,100), font_size=4, line_width=5,)

# image_saver = fr.util.Image_Saver(
#     dir_save=directory_save,
#     overwrite=True,
# )

# image_saver.save_gif(
#     array_images=movie, 
#     name_save='mouse_face_matched', 
#     frame_rate=10.0, 
#     loop=0, 
# )

In [ ]:
## Cast to uint8
movie = [im.astype(np.uint8) for im in images_with_warped_points]
## Add text overlay
movie = fr.helpers.add_text_to_images(movie, [[str(n)] for n in np.arange(len(movie))], position=(50,100), font_size=4, line_width=5,)

image_saver = fr.util.Image_Saver(
    dir_save=directory_save,
    overwrite=True,
)

image_saver.save_gif(
    array_images=movie, 
    name_save='mouse_face_points_matched', 
    frame_rate=6.0, 
    loop=0, 
)